In [1]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import numpy as np
import torch

# add parent dir so importing top level files works in notebook subdir
parent_dir = str(Path().resolve().parent)
sys.path.insert(0, parent_dir)

import qwen_vl

In [ ]:
# graph_root = Path("../output/cholecseg8k/video27_00480_qwen_cat/graph")
graph_root = Path("../output/cholecseg8k/video25_00402_qwen_cat/graph")

In [5]:
feature_files = sorted((graph_root / "c_qwen_feats").glob("*.npy"), key=lambda f: int(f.stem))
qwen_features = [np.load(f) for f in feature_files]
print("n_features", [f.shape[0] for f in qwen_features])
adjacency_matrices = np.load(graph_root / "graph.npy")
print("adjacency_matrices", adjacency_matrices.shape, (adjacency_matrices > 0.0).sum() / adjacency_matrices.size)
centers = np.load(graph_root / "c_centers.npy")
print("centers", centers.shape)
centroids = np.load(graph_root / "c_centroids.npy")
print("centroids", centroids.shape)
extents = np.load(graph_root / "c_extents.npy")
print("extents", extents.shape)

n_features [40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40, 40]
adjacency_matrices (15, 4, 4) 0.5
centers (15, 4, 3)
centroids (15, 4, 3)
extents (15, 4, 3)


In [6]:
adjacency_matrices = adjacency_matrices[::3]
centers = centers[::3]
centroids = centroids[::3]
extents = extents[::3]

In [7]:
# list gpus
torch.cuda.device_count()

2

In [8]:
model, processor = qwen_vl.get_patched_qwen(use_bnb_4bit=False, device_map="cuda:1")

Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

The image processor of type `Qwen2VLImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. Note that this behavior will be extended to all models in a future release.


In [ ]:
# p = "../data/cholecseg8k/preprocessed_ssg/video43/video43_00147/qwen_instance_features/frame_000060_f.npy"
# p = "/home/tumai/nico/surgery-scene-graphs/data/cholecseg8k/preprocessed_ssg/video43/video43_00147/qwen_instance_features/000060_f.npy"
p = "/home/tumai/nico/surgery-scene-graphs/data/cholecseg8k/preprocessed_ssg/video01/video01_00080/qwen_instance_features/000078_f.npy"
f = np.load(p)
qwen_vl.ask_qwen_about_image_features(torch.tensor(f), "Is there a surgical tool visible? If so, what is it touching?", model, processor, system_prompt="You are an expert visceral surgeon.")

In [ ]:
# systemprompt = """
# You are an assistant that answers questions about scene graphs
# of a cholecystectomy procedure.

# A scene graph represents a 3D scene through time as
# a list of graphs, where each graphs corresponds to a
# timestep.
# Each graph contains a set of nodes and edges.
# Each node corresponds to an object present in the scene,
# and associated properties like its position, extent,
# etc. at the respective timestep.
# Edge weights correspond to distances between objects
# at the respective timestep.
# Each object is represented by an image of the
# object.
# The scene graph as a whole is given as a list of
# objects followed by a list of graphs.

# You will be given a scene graph and a question by the user.
# You answer the question ONLY based on the scene graph
# and context provided by this system prompt.

# You answer with nothing but the response to the question.
# You keep the response as concise as possible, while
# answering all aspects the question.
# """

systemprompt = """
You are an expert surgeon that answers questions about
cholecystectomy procedures.

You receive the procedure as a graph, where each
node is an image representing an entiry of the scene,
and a question by the user.

You respond with a concise answer to the question based
and nothing else. You answer only based on the graph.
You NEVER mention the graph, or any of its structure
like nodes, objects, ids, etc. to the user.
"""

In [ ]:
qwen_vl.prompt_with_graph(
    question="At which height is the grasper relative to the gallbladder?",
    node_feats=qwen_features,
    adjacency_matrices=adjacency_matrices,
    node_centers=centers,
    node_centroids=centroids,
    node_extents=extents,
    model=model,
    processor=processor,
    system_prompt=systemprompt,
)

In [ ]:
import os
print("CUDA_VISIBLE_DEVICES:", os.environ.get('CUDA_VISIBLE_DEVICES', 'NOT SET'))
print("Available GPUs to torch:", torch.cuda.device_count())
